# 🌍 Project 30 — Flood Mapping Using NDWI

## 1. Introduce the Project

## Goal

In this project, we will map flooded areas by comparing satellite images **before and after a flood event**.

The main question we want to answer:

> **Which areas became water-covered because of the flood?**

We will create a map showing:

- **Permanent water** → water bodies that existed before the flood (lakes, rivers, reservoirs)
- **Flooded areas** → new water-covered areas that appeared after the flood

---

# Why is this useful in Remote Sensing?

Flood mapping is one of the most common applications of satellite remote sensing.

After a flood, field surveys can be:

- Dangerous
- Expensive
- Slow
- Impossible in remote regions

Satellite images allow researchers and emergency managers to quickly estimate:

- Flood extent
- Affected agricultural land
- Damaged infrastructure
- Changes in rivers and lakes

Applications:

- Disaster management
- Climate change studies
- Hydrology
- Agricultural damage assessment
- Insurance and risk mapping

---

# Main New GEE Skills

In this project, we will combine previous skills:

From previous lessons:

✅ Sentinel-2 filtering  
✅ NDWI calculation  
✅ Thresholding  
✅ Image subtraction  
✅ Change detection  
✅ Masks  

New idea:

> Separating **existing water** from **new flood water**

---

# 2. Big Picture

Before writing code, let's understand the workflow.

```text
Before Flood Image
        ↓
Calculate NDWI
        ↓
Permanent Water Map


After Flood Image
        ↓
Calculate NDWI
        ↓
Flood + Permanent Water Map


        ↓


Compare Before and After


        ↓


New Water Areas


        ↓


Flood Map
````

The logic:

```text
Water after flood
        -
Water before flood
        =
Flooded areas
```

---

# 3. Main Concept

## 3.1 Flood Detection Using Water Indices

Flood water is difficult to detect using normal RGB images because:

* Water can appear dark
* Shadows can look similar to water
* Wet soil can confuse classification

Therefore, we use a spectral index.

Previously, we learned:

# NDWI — Normalized Difference Water Index

Formula:

```text
NDWI = (Green - NIR)
       -------------
       (Green + NIR)
```

For Sentinel-2:

```text
B3 = Green
B8 = NIR
```

So:

```text
NDWI = (B3 - B8)
       -------------
       (B3 + B8)
```

---

## Why does NDWI detect water?

Water behaves differently in different wavelengths.

General idea:

```text
Visible Green Light

        ↓

Water reflects some


Near Infrared (NIR)

        ↓

Water absorbs strongly
```

Therefore:

```text
Green > NIR
```

NDWI becomes positive.

Usually:

```text
High NDWI
      ↓
Water


Low NDWI
      ↓
Vegetation / Soil / Urban areas
```

---

# 4. Step 1 — Define the Study Area

First, we need an area affected by flooding.

For learning, we will define a polygon manually.

```javascript
var studyArea = ee.Geometry.Polygon([
  [
    [51.0, 32.5],
    [51.0, 32.0],
    [51.5, 32.0],
    [51.5, 32.5]
  ]
]);
```

---

## Explanation

`ee.Geometry.Polygon()`

creates a custom study area.

The satellite images will later be clipped to this area.

Workflow:

```text
Large Satellite Image

        ↓

Clip

        ↓

Study Area Only
```

---

# 5. Step 2 — Load Sentinel-2 Images

We need two images:

1. Before flood
2. After flood

Example:

```javascript
var before = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(studyArea)
.filterDate("2024-01-01", "2024-01-30")
.filter(ee.Filter.lt(
  "CLOUDY_PIXEL_PERCENTAGE",
  20
))
.median()
.clip(studyArea);


var after = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(studyArea)
.filterDate("2024-02-01", "2024-02-28")
.filter(ee.Filter.lt(
  "CLOUDY_PIXEL_PERCENTAGE",
  20
))
.median()
.clip(studyArea);
```

---

## Explanation

We already learned this workflow:

```text
ImageCollection

       ↓

filterBounds()

       ↓

filterDate()

       ↓

Cloud filtering

       ↓

median()

       ↓

Single Image
```

---

## Why use `.median()`?

Because several images may exist during a month.

Some may have:

* Clouds
* Noise
* Missing pixels

Median creates a cleaner composite.

---

# 6. Step 3 — Calculate NDWI

Now we calculate water index for both periods.

```javascript
var ndwiBefore = before
  .normalizedDifference(["B3","B8"])
  .rename("NDWI_before");


var ndwiAfter = after
  .normalizedDifference(["B3","B8"])
  .rename("NDWI_after");
```

---

## Explanation

This is the same concept from:

✅ Project 9 — NDWI
✅ Project 10 — Water Change Detection

The difference is:

Previously:

```text
One NDWI image

        ↓

Water map
```

Now:

```text
Two NDWI images

        ↓

Compare them
```

---

# 7. Step 4 — Create Water Masks

An index value alone is not a water map.

We need a threshold.

Example:

```javascript
var waterThreshold = 0.2;
```

Meaning:

```text
NDWI > 0.2

        ↓

Water
```

Create masks:

```javascript
var waterBefore = ndwiBefore.gt(waterThreshold);

var waterAfter = ndwiAfter.gt(waterThreshold);
```

---

## Explanation

`.gt()`

means:

> Greater than

Example:

```javascript
ndwiAfter.gt(0.2)
```

means:

For every pixel:

```text
IF NDWI > 0.2

return 1

otherwise

return 0
```

Result:

```text
Water pixels

1


Non-water pixels

0
```

---

# 8. Step 5 — Identify Permanent Water

Permanent water existed before the flood.

```javascript
var permanentWater = waterBefore;
```

Concept:

```text
Before flood:

Lake
River
Reservoir

=

Permanent Water
```

---

# 9. Step 6 — Detect Flooded Areas

Now we compare before and after.

The idea:

```text
After water

        -

Before water

        ↓

New water
```

Code:

```javascript
var flood = waterAfter.subtract(waterBefore);
```

---

## Understanding Pixel Values

### Pixel A

Before:

```text
0
```

After:

```text
1
```

Calculation:

```text
1 - 0 = 1
```

Meaning:

```text
New water appeared
```

---

### Pixel B

Before:

```text
1
```

After:

```text
1
```

Calculation:

```text
1 - 1 = 0
```

Meaning:

```text
Already water
```

---

### Pixel C

Before:

```text
0
```

After:

```text
0
```

Calculation:

```text
0 - 0 = 0
```

Meaning:

```text
No water
```

---

Therefore:

```text
Flooded Area

=

After Water AND NOT Before Water
```

---

# 10. Step 7 — Display Results

```javascript
Map.centerObject(studyArea,10);


Map.addLayer(
  before,
  {
    bands:["B4","B3","B2"],
    min:0,
    max:3000
  },
  "Before Flood"
);


Map.addLayer(
  after,
  {
    bands:["B4","B3","B2"],
    min:0,
    max:3000
  },
  "After Flood"
);


Map.addLayer(
  permanentWater.selfMask(),
  {
    palette:["blue"]
  },
  "Permanent Water"
);


Map.addLayer(
  flood.selfMask(),
  {
    palette:["red"]
  },
  "Flooded Areas"
);
```

---

## Final Map

```text
Blue

↓

Permanent water


Red

↓

New flood water
```

---

# 11. Complete Workflow Summary

```text
Sentinel-2 Before Flood
          ↓
        NDWI
          ↓
     Water Mask


Sentinel-2 After Flood
          ↓
        NDWI
          ↓
     Water Mask


          ↓


After Water - Before Water


          ↓


Flood Map
```

---

# Complete Code

```javascript
// 1. Study Area

var studyArea = ee.Geometry.Polygon([
  [
    [51.0, 32.5],
    [51.0, 32.0],
    [51.5, 32.0],
    [51.5, 32.5]
  ]
]);


// 2. Before Flood Image

var before = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(studyArea)
.filterDate(
  "2024-01-01",
  "2024-01-30"
)
.filter(
  ee.Filter.lt(
    "CLOUDY_PIXEL_PERCENTAGE",
    20
  )
)
.median()
.clip(studyArea);


// 3. After Flood Image

var after = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(studyArea)
.filterDate(
  "2024-02-01",
  "2024-02-28"
)
.filter(
  ee.Filter.lt(
    "CLOUDY_PIXEL_PERCENTAGE",
    20
  )
)
.median()
.clip(studyArea);


// 4. Calculate NDWI

var ndwiBefore = before
  .normalizedDifference(["B3","B8"])
  .rename("NDWI_before");


var ndwiAfter = after
  .normalizedDifference(["B3","B8"])
  .rename("NDWI_after");


// 5. Water Threshold

var threshold = 0.2;


var waterBefore = ndwiBefore.gt(threshold);

var waterAfter = ndwiAfter.gt(threshold);


// 6. Permanent Water

var permanentWater = waterBefore;


// 7. Flood Detection

var flood = waterAfter.subtract(waterBefore);


// 8. Visualization

Map.centerObject(
  studyArea,
  10
);

Map.addLayer(
  before,
  {
    bands:["B4","B3","B2"],
    min:0,
    max:3000
  },
  "Before Flood"
);

Map.addLayer(
  after,
  {
    bands:["B4","B3","B2"],
    min:0,
    max:3000
  },
  "After Flood"
);

Map.addLayer(
  permanentWater.selfMask(),
  {
    palette:["blue"]
  },
  "Permanent Water"
);

Map.addLayer(
  flood.selfMask(),
  {
    palette:["red"]
  },
  "Flooded Areas"
);
```

---

# Things to Remember

## 1. Flood mapping is a change detection problem

We are not simply finding water.

We are finding:

```text
NEW water
```

The key idea:

```text
After flood water

        -

Before flood water
```

---

## 2. NDWI detects water, not floods

Important distinction:

NDWI tells us:

> Where is water?

Change detection tells us:

> Where did new water appear?

---

## 3. Threshold choice matters

Example:

```text
NDWI > 0.2
```

is only a learning example.

A research project should test thresholds using:

* Field data
* High-resolution imagery
* Regional conditions

---

## 4. Permanent water must be removed

Without removing existing water:

```text
Lake
River
Reservoir
```

would incorrectly appear as flood.

---

# Common Beginner Mistakes

## ❌ Mistake 1: Using only the after-flood image

Example:

```text
After image

      ↓

NDWI

      ↓

Water map
```

Problem:

You cannot distinguish:

* Normal lakes
* Flood water

Correct:

```text
Before + After comparison
```

---

## ❌ Mistake 2: Comparing different seasons

Example:

```text
January vs July
```

Problem:

Vegetation and water naturally change.

Better:

```text
Same season

Before flood

+

After flood
```

---

## ❌ Mistake 3: Choosing an arbitrary threshold without testing

Example:

```text
NDWI > 0.5
```

Problem:

Different environments need different thresholds.

Water in:

* Mountains
* Deserts
* Cities

may behave differently.

---

## ❌ Mistake 4: Ignoring clouds

Clouds can look similar to water.

Always:

* Filter clouds
* Use cloud masking when possible

---

## ❌ Mistake 5: Misinterpreting negative change

Example:

```text
After water - Before water
```

A negative value does not mean "drying".

It may mean:

* Seasonal water decrease
* Cloud problem
* Threshold issue

---

# Your Practice — Project 30.1

## Flood Mapping in a Different Region

Choose a region affected by a real flood.

Examples:

* River flooding
* Lake overflow
* Agricultural flooding

Your task:

Create:

1. Before-flood Sentinel-2 composite
2. After-flood Sentinel-2 composite
3. NDWI for both dates
4. Water masks
5. Permanent water map
6. Flooded area map

Requirements:

* Use `filterDate()`
* Use `.median()`
* Calculate NDWI
* Apply thresholding
* Use change detection

---

# Conceptual Questions

## Question 1

Why do we need a **before flood** image?

Why is an after-flood NDWI map alone not enough?

---

## Question 2

A pixel has:

Before flood:

```text
NDWI = 0.6
```

After flood:

```text
NDWI = 0.7
```

Is this necessarily a flooded pixel?

Explain.

---

## Question 3

Why should flood mapping usually compare images from the same season?

